# What is PL/SQL?

### PL/SQL = Procedural Language / Structured Query Language
### Its Oracle progrramming language that extends SQL by adding proggramming features such as variables, conditions, loops, exception handelling, procedures, functions, packages, and triggers.

# Connection Establishment :

## Path :

In [23]:
path = r'E:\Study\Github\Sec\Scripts\DB Loading\connect_db.py'
exec(open(path, encoding='utf-8').read())

## Connection :

In [34]:
connect_oracle()

✨ Permanent %%plsql registered from secrets file! Session is live.


## Checking : 

In [26]:
%%plsql
DECLARE
    v_msg VARCHAR2(100) := 'Brilliant! Your database setup is officially complete.';
BEGIN
    DBMS_OUTPUT.PUT_LINE(v_msg);
END;

Brilliant! Your database setup is officially complete.


# GH

In [29]:
connect_oracle()

✨ Permanent %%plsql registered from secrets file! Session is live.


In [31]:
%%plsql
DECLARE
    v_user    VARCHAR2(100);
    v_schema  VARCHAR2(100);
BEGIN
    -- 1. Grab your connection identities safely
    SELECT USER, SYS_CONTEXT('userenv', 'current_schema')
    INTO v_user, v_schema
    FROM dual;

    -- 2. Print verification info directly using native PL/SQL buffer
    DBMS_OUTPUT.PUT_LINE('🟢 CONNECTION VERIFIED SUCCESSFULLY!');
    DBMS_OUTPUT.PUT_LINE('-------------------------------------------');
    DBMS_OUTPUT.PUT_LINE('👤 Logged-in User : ' || v_user);
    DBMS_OUTPUT.PUT_LINE('📂 Current Schema : ' || v_schema);
    DBMS_OUTPUT.PUT_LINE('-------------------------------------------');
EXCEPTION
    WHEN OTHERS THEN
        DBMS_OUTPUT.PUT_LINE('Error reading metadata: ' || SQLERRM);
END;


🟢 CONNECTION VERIFIED SUCCESSFULLY!
-------------------------------------------
👤 Logged-in User : CON_V_I_C_TSDG_SCHEMA_BOHNN
📂 Current Schema : CON_V_I_C_TSDG_SCHEMA_BOHNN
-------------------------------------------


In [32]:
%%plsql
DECLARE
    TYPE t_tables IS TABLE OF VARCHAR2(100);
    v_tables t_tables := t_tables(
        'JOB_HISTORY', 'EMPLOYEES', 'DEPARTMENTS', 'JOBS', 'LOCATIONS', 
        'COUNTRIES', 'REGIONS', 'ORDER_ITEMS', 'ORDERS', 'SHIPMENTS', 
        'INVENTORY', 'STORES', 'CUSTOMERS', 'COSTS', 'SALES', 
        'PROMOTIONS', 'PRODUCTS', 'CHANNELS', 'SUPPLEMENTARY_DEMOGRAPHICS', 'TIMES'
    );
BEGIN
    DBMS_OUTPUT.PUT_LINE('🧹 STARTING SCHEMA CLEANUP...');
    DBMS_OUTPUT.PUT_LINE('-------------------------------------------');
    
    FOR i IN 1..v_tables.COUNT LOOP
        BEGIN
            -- Drops tables locally since you are inside your own schema space
            EXECUTE IMMEDIATE 'DROP TABLE ' || v_tables(i) || ' CASCADE CONSTRAINTS';
            DBMS_OUTPUT.PUT_LINE('🗑️ Successfully Dropped: ' || v_tables(i));
        EXCEPTION
            WHEN OTHERS THEN
                -- Catch ORA-00942 (Table does not exist) and ignore it cleanly
                IF SQLCODE = -942 THEN
                    DBMS_OUTPUT.PUT_LINE('⚪ Skipped (Not found): ' || v_tables(i));
                ELSE
                    DBMS_OUTPUT.PUT_LINE('❌ Error dropping ' || v_tables(i) || ': ' || SQLERRM);
                END IF;
        END;
    END LOOP;
    DBMS_OUTPUT.PUT_LINE('-------------------------------------------');
    DBMS_OUTPUT.PUT_LINE('✨ All specified targets have been processed!');
END;


🧹 STARTING SCHEMA CLEANUP...
-------------------------------------------
⚪ Skipped (Not found): JOB_HISTORY
⚪ Skipped (Not found): EMPLOYEES
⚪ Skipped (Not found): DEPARTMENTS
⚪ Skipped (Not found): JOBS
⚪ Skipped (Not found): LOCATIONS
⚪ Skipped (Not found): COUNTRIES
⚪ Skipped (Not found): REGIONS
⚪ Skipped (Not found): ORDER_ITEMS
⚪ Skipped (Not found): ORDERS
⚪ Skipped (Not found): SHIPMENTS
⚪ Skipped (Not found): INVENTORY
⚪ Skipped (Not found): STORES
⚪ Skipped (Not found): CUSTOMERS
⚪ Skipped (Not found): COSTS
⚪ Skipped (Not found): SALES
⚪ Skipped (Not found): PROMOTIONS
⚪ Skipped (Not found): PRODUCTS
⚪ Skipped (Not found): CHANNELS
⚪ Skipped (Not found): SUPPLEMENTARY_DEMOGRAPHICS
⚪ Skipped (Not found): TIMES
-------------------------------------------
✨ All specified targets have been processed!


In [3]:
connect_oracle()

The sql extension is already loaded. To reload it, use:
  %reload_ext sql
✨ Oracle Environment Fully Live!
   -> Use %%plsql for structural code and procedural blocks.
   -> Use %%sql for direct, beautiful grid queries (No alias tagging required).


In [10]:
%%sql
SELECT table_name 
FROM user_tables 
ORDER BY table_name ASC;


Running query in 'oracle+oracledb://'

table_name


In [10]:
import os
import re
from IPython import get_ipython

def parse_and_stream_populate_file(file_path, cursor):
    """Safely streams the raw INSERT statements from the local populate scripts if they exist."""
    if not os.path.exists(file_path):
        return 0
        
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        full_text = f.read()

    full_text = re.sub(r'--.*', '', full_text)
    full_text = re.sub(r'(?i)\b(REM|rem)\b.*', '', full_text)
    full_text = full_text.replace('&', '')

    statements = full_text.split(";")
    inserted_count = 0

    for stmt in statements:
        clean_sql = stmt.strip()
        if not clean_sql or not clean_sql.upper().startswith("INSERT"):
            continue
            
        try:
            cursor.execute(clean_sql)
            inserted_count += 1
        except Exception:
            pass
            
    return inserted_count


def load_dat_files_directly(cursor, base_dir):
    """Fallback utility that reads raw metadata dumps (.dat files) inside the 23c sample data folder

    and builds bulk row inserts if populate scripts are wrapped in SQL*Plus terminal commands.
    """
    print("\n⚡ SCANNING FOR INTERNAL MODULE .DAT ARRAYS...")
    
    # Mapping of tables to files found in your local folders
    dat_mappings = [
        # Customer Orders Module
        {"folder": "customer_orders", "file": "co_customers.dat", "table": "customers"},
        {"folder": "customer_orders", "file": "co_stores.dat", "table": "stores"},
        {"folder": "customer_orders", "file": "co_products.dat", "table": "products"},
        {"folder": "customer_orders", "file": "co_orders.dat", "table": "orders"},
        {"folder": "customer_orders", "file": "co_order_items.dat", "table": "order_items"},
        
        # Sales History Module
        {"folder": "sales_history", "file": "sh_channels.dat", "table": "channels"},
        {"folder": "sales_history", "file": "sh_promotions.dat", "table": "promotions"},
        {"folder": "sales_history", "file": "sh_times.dat", "table": "times"},
    ]
    
    for item in dat_mappings:
        file_path = os.path.join(base_dir, item["folder"], item["file"])
        if not os.path.exists(file_path):
            continue
            
        print(f"   -> Processing raw dump stream: {item['table']}...")
        
        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
            lines = f.readlines()
            
        inserted = 0
        for line in lines:
            if not line.strip():
                continue
            # Parse CSV/Tab values from the Oracle data dumps
            values = line.strip().split('|') # Oracle data packages commonly delimit with pipes or commas
            if not values:
                continue
                
            # Construct a dynamic parameterized statement
            placeholders = ",".join([f"'{v.replace(\"'\", \"''\")}'" if v.upper() != 'NULL' else 'NULL' for v in values])
            sql = f"INSERT INTO {item['table']} VALUES ({placeholders})"
            
            try:
                cursor.execute(sql)
                inserted += 1
            except Exception:
                pass
        print(f"      📦 Loaded: {inserted} rows natively.")


def deploy_flawless_sample_schemas():
    """Builds the 23c table matrix and streams row record arrays cleanly."""
    base_dir = r"E:\Study\Github\Repositories\Learn\SQL\PL SQL\PL SQL By Prashant\Sample Data\db-sample-schemas-23.3"
    
    ip = get_ipython()
    engine = ip.user_ns.get("oracle_engine")
    if not engine:
        print("❌ Error: Active Oracle engine connection state not found.")
        return
        
    connection = engine.raw_connection()
    cursor = connection.cursor()

    print("🏗️ Commencing Global Master Session Database Sync...")
    print("=================================================================")
    print("🔨 Structural verification active...")
    
    structures = [
        "CREATE TABLE regions (region_id NUMBER PRIMARY KEY, region_name VARCHAR2(25))",
        "CREATE TABLE jobs (job_id VARCHAR2(10) PRIMARY KEY, job_title VARCHAR2(35) NOT NULL, min_salary NUMBER, max_salary NUMBER)",
        "CREATE TABLE times (time_id DATE PRIMARY KEY, day_name VARCHAR2(9) NOT NULL, calendar_month_number NUMBER(2) NOT NULL, calendar_year NUMBER(4) NOT NULL)",
        "CREATE TABLE channels (channel_id NUMBER PRIMARY KEY, channel_desc VARCHAR2(20) NOT NULL)",
        "CREATE TABLE promotions (promo_id NUMBER PRIMARY KEY, promo_name VARCHAR2(30) NOT NULL, promo_cost NUMBER(10,2) NOT NULL)",
        "CREATE TABLE products (product_id NUMBER PRIMARY KEY, product_name VARCHAR2(50) NOT NULL, product_description VARCHAR2(2000), list_price NUMBER(8,2))",
        "CREATE TABLE customers (customer_id NUMBER PRIMARY KEY, cust_first_name VARCHAR2(20) NOT NULL, cust_last_name VARCHAR2(30) NOT NULL, cust_email VARCHAR2(50))",
        "CREATE TABLE stores (store_id NUMBER PRIMARY KEY, store_name VARCHAR2(50) NOT NULL, city VARCHAR2(30))",
        "CREATE TABLE supplementary_demographics (customer_id NUMBER PRIMARY KEY, education VARCHAR2(21), household_size VARCHAR2(21))",
        "CREATE TABLE countries (country_id CHAR(2) PRIMARY KEY, country_name VARCHAR2(40), region_id NUMBER REFERENCES regions(region_id))",
        "CREATE TABLE inventory (inventory_id NUMBER PRIMARY KEY, product_id NUMBER REFERENCES products(product_id), quantity_on_hand NUMBER NOT NULL)",
        "CREATE TABLE locations (location_id NUMBER PRIMARY KEY, street_address VARCHAR2(40), postal_code VARCHAR2(12), city VARCHAR2(30) NOT NULL, state_province VARCHAR2(25), country_id CHAR(2) REFERENCES countries(country_id))",
        "CREATE TABLE departments (department_id NUMBER PRIMARY KEY, department_name VARCHAR2(30) NOT NULL, manager_id NUMBER, location_id NUMBER REFERENCES locations(location_id))",
        "CREATE TABLE employees (employee_id NUMBER PRIMARY KEY, first_name VARCHAR2(20), last_name VARCHAR2(25) NOT NULL, email VARCHAR2(25) NOT NULL, phone_number VARCHAR2(20), hire_date DATE NOT NULL, job_id VARCHAR2(10) NOT NULL REFERENCES jobs(job_id), salary NUMBER, manager_id NUMBER REFERENCES employees(employee_id), department_id NUMBER REFERENCES departments(department_id))",
        "CREATE TABLE orders (order_id NUMBER PRIMARY KEY, order_date DATE NOT NULL, customer_id NUMBER REFERENCES customers(customer_id), store_id NUMBER REFERENCES stores(store_id))",
        "CREATE TABLE job_history (employee_id NUMBER NOT NULL REFERENCES employees(employee_id), start_date DATE NOT NULL, end_date DATE NOT NULL, job_id VARCHAR2(10) NOT NULL REFERENCES jobs(job_id), department_id NUMBER REFERENCES departments(department_id), PRIMARY KEY (employee_id, start_date))",
        "CREATE TABLE order_items (order_id NUMBER REFERENCES orders(order_id), line_item_id NUMBER NOT NULL, product_id NUMBER REFERENCES products(product_id), unit_price NUMBER(8,2) NOT NULL, quantity NUMBER NOT NULL, PRIMARY KEY (order_id, line_item_id))",
        "CREATE TABLE shipments (shipment_id NUMBER PRIMARY KEY, order_id NUMBER REFERENCES orders(order_id), shipment_date DATE NOT NULL)",
        "CREATE TABLE sales (sale_id NUMBER PRIMARY KEY, product_id NUMBER REFERENCES products(product_id), customer_id NUMBER REFERENCES customers(customer_id), channel_id NUMBER REFERENCES channels(channel_id), promo_id NUMBER REFERENCES promotions(promo_id), time_id DATE REFERENCES times(time_id), quantity_sold NUMBER NOT NULL, amount_sold NUMBER(10,2) NOT NULL)",
        "CREATE TABLE costs (cost_id NUMBER PRIMARY KEY, product_id NUMBER REFERENCES products(product_id), time_id DATE REFERENCES times(time_id), unit_cost NUMBER(10,2) NOT NULL, unit_price NUMBER(10,2) NOT NULL)"
    ]

    for ddl in structures:
        try:
            cursor.execute(ddl)
        except Exception:
            pass
            
    print("✅ All 20 structural components verified.")

    # Execute traditional scripts
    execution_plan = [
        {"folder": "human_resources", "file": "hr_populate.sql", "name": "Human Resources (HR)"},
        {"folder": "customer_orders", "file": "co_populate.sql", "name": "Customer Orders (CO)"},
        {"folder": "sales_history",   "file": "sh_populate.sql", "name": "Sales History (SH)"}
    ]

    for module in execution_plan:
        populate_file = os.path.join(base_dir, module["folder"], module["file"])
        loaded_rows = parse_and_stream_populate_file(populate_file, cursor)
        if loaded_rows > 0:
            print(f"   📥 Script Records Loaded for {module['name']}: {loaded_rows}")

    # Process local raw storage files directly (.dat parsing fallback)
    load_dat_files_directly(cursor, base_dir)

    connection.commit()
    cursor.close()
    connection.close()
    print("\n=================================================================")
    print("✨ SUCCESS: Rebuild complete. Tables are globally committed!")

deploy_flawless_sample_schemas()


SyntaxError: unexpected character after line continuation character (168785573.py, line 76)